#### Generate tf.records from the lsdb file compatible for the model and performs preprocessing.

In [4]:
import os
import numpy as np
import tensorflow as tf 
from lsdb import read_hats
import matplotlib.pyplot as plt
from dask.distributed import Client
from astra.bands.bands import ztf_band
from astra.utils.helper import standardize, filter_by_class
from astra.src.dataset import create_dataset

In [3]:
#
# Read catalog
#
read_catalog = read_hats('/media3/majumder/dataset/cepheids/hats/zubercal_vcep', )
path_to_store="/media3/majumder/dataset/cepheids_new/"
#
#
#
catalog_compute = read_catalog._ddf.map_partitions(create_dataset, 
                                                        target=path_to_store,
                                                        bands=ztf_band,
                                                        seed=42,
                                                        min_detec=100,
                                                        train_size=.80,
                                                        max_lcs_per_chunk=200)

with Client() as client:
    catalog_compute.compute(scheduler='processes')


print("\nDone!")

    

2025-09-23 22:56:02.090899: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-23 22:56:02.094976: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-23 22:56:02.106879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758660962.127234 2773828 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758660962.133406 2773828 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758660962.149427 2773828 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin


Done!


#### Generate histogram of the lightcurve for each ZTF-band to determine "min_detec" param in the create_dataset()

In [ ]:
#
#
#
hist = dict()
labels = list()
counter = 0
path = f"/media3/majumder/dataset/lyrae_cep/val/"
#
#
#
glob_pattern = os.path.join(path, 'partition_*', '*', 'chunk_*.record')
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
num_files_found = tf.data.experimental.cardinality(filenames)
print(f"Number of files found: {num_files_found.numpy()}")
#
#
#
for k in ztf_band.keys():
    hist[k] = list()
#
dataset = tf.data.TFRecordDataset(filenames)
#
#
#
for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    #
    last_index = example.context.feature['last_index'].int64_list.value
    time = example.feature_lists.feature_list['dim_0'].feature[0].float_list.value
    mag = example.feature_lists.feature_list['dim_1'].feature[0].float_list.value
    band_sorted = example.feature_lists.feature_list['dim_3'].feature[0].float_list.value
    last_index = example.context.feature['last_index'].int64_list.value
    id_ = example.context.feature['id'].int64_list.value[0]
    
    j=0
    for i, l in enumerate(hist.keys()):
        #
        temp = last_index[i]+1
        length = temp-j
        hist[l].append(length)
        j = temp
        
print(hist)

In [ ]:
#
#
#
plt.figure(figsize=(8, 10))
plt.suptitle(f"Histogram displaying the lengths of light curves for each ZTF filter")
for i , k in enumerate(hist.keys()):
    # 
    plt.subplot(3, 1, i+1)
    plt.hist(hist[k], bins=30, color="#3492eb", rwidth=0.9)
    plt.xlim(min(hist[k])-500, max(hist[k])+100)
    plt.title(f"ZTF-{k} filter")
    plt.subplots_adjust(hspace=0.5)
    plt.xlabel("# of detections in each light curve")
# plt.savefig("./cephids/length.png")

#### Determine the ZTF mag_limit and mag_saturation value

In [ ]:
#
#
#
magnitude = list()
labels = list()
chunks = list()
c=0
path = f"/media3/majumder/dataset/lyrae_cep/val/"
#
#
#
glob_pattern = os.path.join(path, 'partition_*', '*', 'chunk_*.record')
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
num_files_found = tf.data.experimental.cardinality(filenames)
print(f"Number of files found: {num_files_found.numpy()}")
#
#
#
dataset = tf.data.TFRecordDataset(filenames)
#
#

for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    # Convert to TensorFlow tensors
    mag = tf.convert_to_tensor(example.feature_lists.feature_list['dim_1'].feature[0].float_list.value, dtype=tf.float64)
    magerr = tf.convert_to_tensor(example.feature_lists.feature_list['dim_2'].feature[0].float_list.value, dtype=tf.float64)
    c+=1
    #
    new_mag, _ = standardize(mag, magerr)
    magnitude.extend(new_mag.numpy())
        
print(c)
# Total available data: 3941899925

In [ ]:
# Calculate 1st and 99th percentiles
percentile_1 = np.percentile(magnitude, 1)
percentile_99 = np.percentile(magnitude, 99)
# Filter data within the 99% range
magnitude_np = np.array(magnitude)
filtered_data = magnitude_np[(magnitude_np >= percentile_1) & (magnitude_np <= percentile_99)]
#
# The min value represents the brightest detection and the max value represents the faintest detection
# mag_saturation = min(filtered_data)
# mag_limit = max(filtered_data)
print(len(filtered_data), min(filtered_data), max(filtered_data))

# The values of the above parameters: 3863061925 -0.6958509378753988 1.5205410620126045

#### Generate data for Finetuning

In [ ]:
path_to_read = "/media3/majumder/dataset/cepheids/val/"
path_to_store = "/media3/majumder/dataset/cepheids/finetuning/"
objects_per_chunk = 100
generate_data_finetuning(path_to_read, path_to_store, objects_per_chunk)

#### Check data in the finetuning folder

In [ ]:
#
#
#
mag=[]
c=0
# filenames = list()

path = f"/media3/majumder/dataset/lyrae_cep/finetuning/"
#
glob_pattern = os.path.join(path, 'chunk_*.record')
filenames = tf.data.Dataset.list_files(glob_pattern, shuffle=False)
num_files_found = tf.data.experimental.cardinality(filenames)
print(f"Number of files found: {num_files_found.numpy()}")
# 
dataset = tf.data.TFRecordDataset(filenames)
#
for rec in dataset:
    #
    example = tf.train.SequenceExample()
    example.ParseFromString(rec.numpy())
    # print(example)
    c+=1
    
print(c)    